In [ ]:
import sys, importlib, os

_src_training = os.path.dirname(os.path.abspath("training.ipynb"))
_src = os.path.dirname(_src_training)
for _p in [_src_training, _src]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import infrastructure, pipeline, classifier, evaluation
for _mod in [infrastructure, pipeline, classifier, evaluation]:
    importlib.reload(_mod)

from infrastructure import clean_neo4j_db, clean_kafka_topics, delete_test_neo4j_nodes, verify_concept_created
from pipeline import train_mnist, remove_concept, retrain_concept
from evaluation import test_mnist_all
from classifier import classify_image
import json, uuid
print("Modules loaded.")

In [ ]:
classes_to_subclasses = {
    0: [1],
    1: [1, 3],
    2: [1, 2],
    3: [1],
    4: [1, 2],
    5: [1],
    6: [1],
    7: [1],
    8: [1],
    9: [2],
}

In [ ]:
# clean_neo4j_db()

# for class_num in classes_to_subclasses:
#     for subclass in classes_to_subclasses[class_num]:
#         train_mnist(class_number=class_num, subclass=subclass, is_prepared_samples=True, with_concept_creation=True)

In [ ]:
# ── Restore concepts from JSON snapshot into Neo4j ──
# Uncomment and run this cell to load your colleague's concepts
# This will REPLACE all existing concepts in Neo4j

import json as _json
from neo4j import GraphDatabase

SNAPSHOT_PATH = "../../concept_graphs.json"

_uri = os.environ.get("NEO4J_DSN", "bolt://localhost:7687")
_user = os.environ.get("NEO4J_USER", "neo4j")
_pw = os.environ.get("NEO4J_PASSWORD", "111122223333")

with open(SNAPSHOT_PATH) as f:
    snapshot = _json.load(f)

driver = GraphDatabase.driver(_uri, auth=(_user, _pw))

# Clean existing concepts
with driver.session() as session:
    session.run("MATCH (n) WHERE n.concept_id IS NOT NULL DETACH DELETE n")
    print("Cleared existing concepts from Neo4j.")

for cid, gdata in snapshot.items():
    nodes = gdata.get("nodes", [])
    edges = gdata.get("edges", gdata.get("links", []))

    # Map old node IDs to new ones
    id_map = {}
    with driver.session() as session:
        for node in nodes:
            labels = node.get("labels", [])
            label_str = ":".join(labels)
            props = {}
            for k, v in node.items():
                if k in ("labels",):
                    continue
                if isinstance(v, (dict, list)):
                    props[k] = _json.dumps(v)
                else:
                    props[k] = v
            old_id = node.get("id", node.get("uuid"))
            result = session.run(
                f"CREATE (n:{label_str} $props) RETURN elementId(n) AS nid",
                props=props
            )
            new_id = result.single()["nid"]
            id_map[old_id] = new_id

        for edge in edges:
            src = id_map.get(edge["source"])
            tgt = id_map.get(edge["target"])
            rel_type = edge.get("type", "CONNECTED_TO")
            if src and tgt:
                session.run(
                    f"MATCH (a), (b) WHERE elementId(a) = $src AND elementId(b) = $tgt "
                    f"CREATE (a)-[:{rel_type}]->(b)",
                    src=src, tgt=tgt
                )

    print(f"  Restored concept {cid}: {len(nodes)} nodes, {len(edges)} edges")

driver.close()
print(f"\nDone. Restored {len(snapshot)} concepts from {SNAPSHOT_PATH}")

In [ ]:
# ── Visualize concept graphs (from Neo4j or JSON snapshot) ──
import json as _json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from neo4j import GraphDatabase

# Set SOURCE: "neo4j" to load from database, or path to JSON file
SOURCE = "neo4j"
# SOURCE = "../../concept_graphs.json"  # uncomment to load colleague's snapshot

LABEL_COLORS = {
    "StartPoint": "#2ecc71", "EndPoint": "#e74c3c",
    "IntersectionPoint": "#f39c12", "CornerPoint": "#9b59b6",
    "Point": "#3498db", "Vector": "#95a5a6",
}


def get_node_color(labels):
    for l in ["StartPoint", "EndPoint", "IntersectionPoint", "CornerPoint", "Point", "Vector"]:
        if l in labels:
            return LABEL_COLORS[l]
    return "#bdc3c7"

def get_label_abbrev(labels):
    for l, abbr in [("StartPoint","StP"), ("EndPoint","EnP"), ("IntersectionPoint","IntP"),
                     ("CornerPoint","CrP"), ("HorizontalVector","H"), ("VerticalVector","V"), ("Vector","Vec")]:
        if l in labels:
            return abbr
    return "P"

def parse_coord(val):
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        try: return float(val)
        except ValueError: pass
        try:
            d = _json.loads(val)
            if isinstance(d, dict) and "center" in d:
                return float(d["center"])
        except Exception: pass
    if isinstance(val, dict) and "center" in val:
        return float(val["center"])
    return None

def load_from_neo4j():
    uri = os.environ.get("NEO4J_DSN", "bolt://localhost:7687")
    user = os.environ.get("NEO4J_USER", "neo4j")
    pw = os.environ.get("NEO4J_PASSWORD", "111122223333")
    driver = GraphDatabase.driver(uri, auth=(user, pw))
    with driver.session() as session:
        cids = [r["cid"] for r in session.run(
            "MATCH (n) WHERE n.concept_id IS NOT NULL RETURN DISTINCT n.concept_id AS cid ORDER BY cid")]
    graphs = {}
    for cid in cids:
        with driver.session() as session:
            records = list(session.run("""
                MATCH (n) WHERE n.concept_id = $cid
                OPTIONAL MATCH (n)-[r]-(m {concept_id: $cid})
                RETURN elementId(n) AS node_id, labels(n) AS labels, properties(n) AS props,
                       type(r) AS rel_type, elementId(m) AS target_id
            """, cid=cid))
        G = nx.Graph()
        for rec in records:
            nid = rec["node_id"]
            if nid not in G:
                G.add_node(nid, labels=set(rec["labels"]), **rec["props"])
            if rec["target_id"] and rec["target_id"] != nid:
                G.add_edge(nid, rec["target_id"])
        graphs[cid] = G
    driver.close()
    return graphs

def load_from_json(path):
    with open(path) as f:
        data = _json.load(f)
    graphs = {}
    for cid, gdata in data.items():
        G = nx.Graph()
        for n in gdata.get("nodes", []):
            nid = n.get("id", n.get("uuid", str(id(n))))
            labels = set(n.get("labels", []))
            G.add_node(nid, labels=labels, **{k: v for k, v in n.items() if k not in ("id", "labels")})
        for e in gdata.get("edges", gdata.get("links", [])):
            G.add_edge(e["source"], e["target"])
        graphs[cid] = G
    return graphs

# Load concepts
if SOURCE == "neo4j":
    concept_graphs = load_from_neo4j()
    print(f"Loaded {len(concept_graphs)} concepts from Neo4j")
else:
    concept_graphs = load_from_json(SOURCE)
    print(f"Loaded {len(concept_graphs)} concepts from {SOURCE}")

for cid, G in concept_graphs.items():
    print(f"  {cid}: {len(G.nodes)} nodes, {len(G.edges)} edges")

# Compute positions
def compute_positions(G):
    pos = {}
    for node, data in G.nodes(data=True):
        nx_val = parse_coord(data.get("normalized_x"))
        ny_val = parse_coord(data.get("normalized_y"))
        if nx_val is not None and ny_val is not None:
            pos[node] = (nx_val, -ny_val)
    for node in G.nodes:
        if node in pos:
            continue
        neighbors = [n for n in G.neighbors(node) if n in pos]
        if neighbors:
            ax = sum(pos[n][0] for n in neighbors) / len(neighbors)
            ay = sum(pos[n][1] for n in neighbors) / len(neighbors)
            pos[node] = (ax + 0.03 * (hash(str(node)) % 5 - 2), ay + 0.03 * (hash(str(node)) % 7 - 3))
        else:
            pos[node] = (0, 0)
    return pos

# Plot
concept_ids = sorted(concept_graphs.keys())
n = len(concept_ids)
cols = min(4, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows == 1 and cols == 1: axes = [[axes]]
elif rows == 1: axes = [axes]
elif cols == 1: axes = [[a] for a in axes]

for idx, cid in enumerate(concept_ids):
    ax = axes[idx // cols][idx % cols]
    G = concept_graphs[cid]
    if not G.nodes:
        ax.set_title(f"Concept {cid}\n(empty)"); ax.axis("off"); continue

    pos = compute_positions(G)

    points = [n for n in G.nodes if "Vector" not in G.nodes[n].get("labels", set())]
    vectors = [n for n in G.nodes if "Vector" in G.nodes[n].get("labels", set())]

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#7f8c8d", width=2, alpha=0.6)
    if points:
        nx.draw_networkx_nodes(G, pos, nodelist=points, ax=ax, node_shape="o", node_size=500,
                               node_color=[get_node_color(G.nodes[n].get("labels", set())) for n in points])
    if vectors:
        nx.draw_networkx_nodes(G, pos, nodelist=vectors, ax=ax, node_shape="s", node_size=200,
                               node_color=[get_node_color(G.nodes[n].get("labels", set())) for n in vectors])

    labels = {n: get_label_abbrev(G.nodes[n].get("labels", set())) for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7, font_color="white", font_weight="bold")

    ax.set_title(f"Concept {cid}\n({len(points)}P + {len(vectors)}V, {len(G.edges)}E)", fontsize=11, fontweight="bold")
    ax.axis("off")

for idx in range(n, rows * cols):
    axes[idx // cols][idx % cols].axis("off")

legend_patches = [mpatches.Patch(color=c, label=l) for l, c in LABEL_COLORS.items()]
fig.legend(handles=legend_patches, loc="lower center", ncol=len(legend_patches), fontsize=10)
plt.suptitle(f"Concept Graphs ({SOURCE})", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "comparison_method": "fgw",
    "fgw_alpha": 0.6,
    "property_normalizers": {
        "normalized_x": 3.0,
        "normalized_y": 3.0,
        "horizontal_direction": 2.0,
        "vertical_direction": 2.0,
        "cycle_count": 1.0,
        "angle_with_ox": 45,
    },
}
results, y_true, y_pred, run_dir = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params,
    sample_fraction=1.0,
    description="Testing after retraining with the new concept formation with fgw method",
)

In [ ]:
# ── Experiment: Test on MNIST-all (10k images from datasets/mnist_all/) ──
# Copies images into tests/ so nuclio containers can access them via volume mount
import os, pathlib, shutil

_project_root = pathlib.Path(os.path.abspath("training.ipynb")).parent.parent.parent
_src_dir = _project_root / "datasets" / "mnist_all"
_dst_dir = _project_root / "tests" / "mnist_all"

# Remove old symlink if it exists, then copy
if _dst_dir.is_symlink():
    _dst_dir.unlink()

if not _dst_dir.exists():
    print(f"Copying {_src_dir} -> {_dst_dir} ...")
    shutil.copytree(_src_dir, _dst_dir)
    print(f"Done. Copied {sum(1 for _ in _dst_dir.rglob('*.png'))} images.")
    
else:
    print(f"Directory already exists: {_dst_dir}")

params_mnist_all = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "comparison_method": "fgw",
    "fgw_alpha": 0.6,
    "property_normalizers": {
        "normalized_x": 3.0,
        "normalized_y": 3.0,
        "horizontal_direction": 2.0,
        "vertical_direction": 2.0,
        "cycle_count": 1.0,
        "angle_with_ox": 45,
    },
}

results_mnist_all, y_true_mnist_all, y_pred_mnist_all, run_dir_mnist_all = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params_mnist_all,
    sample_fraction=1.0,
    description="Experiment: MNIST-all 10k test set from datasets/mnist_all",
    local_path_template=str(_dst_dir / "{cls}"),
    nuclio_volume_path_template="/opt/nuclio/shared_storage/mnist_all/{cls}",
)

In [ ]:
# ── Experiment: Test on manifest "complete" images only ──
# Filters datasets/manifest.csv for structure=="complete", copies only those
# images into tests/ so nuclio containers can access them.
import os, pathlib, shutil
import pandas as pd

_project_root = pathlib.Path(os.path.abspath("training.ipynb")).parent.parent.parent
_datasets_dir = _project_root / "datasets"
_manifest = pd.read_csv(_datasets_dir / "manifest.csv")

# Filter for complete images only
_complete = _manifest[_manifest["structure"] == "complete"].copy()
print(f"Total 'complete' images in manifest: {len(_complete)}")
print(f"Per class:\n{_complete['class'].value_counts().sort_index().to_string()}")

# Build a filtered directory with copies: tests/manifest_complete/{cls}/
_filtered_root = _project_root / "tests" / "manifest_complete"
if _filtered_root.exists():
    shutil.rmtree(_filtered_root)

copied = 0
for _, row in _complete.iterrows():
    cls = int(row["class"])
    src_image = _datasets_dir / row["image_path"]
    if not src_image.exists():
        continue
    dst_dir = _filtered_root / str(cls)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_file = dst_dir / src_image.name
    if not dst_file.exists():
        shutil.copy2(src_image, dst_file)
        copied += 1

print(f"Copied {copied} images to {_filtered_root}")
for d in sorted(_filtered_root.iterdir()):
    if d.is_dir():
        print(f"  Class {d.name}: {len(list(d.iterdir()))} images")

params_complete = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "comparison_method": "fgw",
    "fgw_alpha": 0.6,
    "property_normalizers": {
        "normalized_x": 3.0,
        "normalized_y": 3.0,
        "horizontal_direction": 2.0,
        "vertical_direction": 2.0,
        "cycle_count": 1.0,
        "angle_with_ox": 45,
    },
}

results_complete, y_true_complete, y_pred_complete, run_dir_complete = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params_complete,
    sample_fraction=1.0,
    description="Experiment: manifest 'complete' images only (curated test set)",
    local_path_template=str(_filtered_root / "{cls}"),
    nuclio_volume_path_template="/opt/nuclio/shared_storage/manifest_complete/{cls}",
)

In [ ]:
import optuna
import mlflow
import random as _random
import torch
from sklearn.metrics import accuracy_score
from optuna_integration import BoTorchSampler
from botorch.models import SaasFullyBayesianSingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.utils.transforms import normalize
from botorch.acquisition import LogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.fit import fit_fully_bayesian_model_nuts

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Configuration ──────────────────────────────────────────
SAMPLE_FRACTION = 0.50     # 50% proxy, remaining 50% as implicit holdout
K_FOLDS = 4                # fold rotation for anti-overfitting
WEIGHT_THRESHOLD = 0.05    # weights below this → feature excluded

PHASE1_TRIALS = 200        # SAASBO: feature discovery (GP saturates beyond ~200)
PHASE2_TRIALS = 3000       # CMA-ES: refinement (≥100×D for convergence)
STUDY_NAME = "naturalagi-feature-weights-v2"

OPTUNA_MLFLOW_EXPERIMENT = STUDY_NAME

# ── Fixed params from best GED trial (#167, 78.4%) ────────
_BEST_GED_PARAMS = {
    "ged_timeout": 8.72,
    "skeletonization_threshold": 140,
    "simplification_epsilon": 4.55,
    "node_costs": {
        "NO_COST": 0.0,
        "MINOR": 0.254,
        "GENERAL": 0.450,
        "SEVERE": 0.726,
        "NO_MATCH": 1.304,
        "IMPOSSIBLE": 6.681,
    },
}

# ── All features with fixed normalizers ────────────────────
_ALL_FEATURES = {
    # Base (spatial + direction + topology)
    "normalized_x":               3.75,
    "normalized_y":               3.67,
    "horizontal_direction":       3.11,
    "cycle_count":                3.28,
    "angle_with_ox":              279.4,
    # Spatial
    "distance_to_centroid":       1.0,
    # Degree / topology
    "node_degree":                4.0,
    "is_endpoint":                1.0,
    "is_junction":                1.0,
    "is_corner":                  1.0,
    "is_on_cycle":                1.0,
    # Angle
    "junction_angle_min":         180.0,
    "junction_angle_max":         180.0,
    "junction_angle_mean":        180.0,
    # Geometric / length
    "tortuosity":                 2.0,
    "normalized_length":          1.0,
    "length_ratio_to_max":        1.0,
    # Branch type
    "branch_type":                6.0,
    "connects_cycle_nodes":       1.0,
    # Centrality
    "betweenness_centrality":     1.0,
    "closeness_centrality":       1.0,
    "eccentricity":               10.0,
    "pagerank":                   1.0,
    # Neighborhood context
    "avg_neighbor_vector_length": 50.0,
    "neighbor_endpoint_count":    3.0,
    "neighbor_junction_count":    3.0,
    "neighbor_corner_count":      3.0,
}


def saasbo_candidates_func(
    train_x: torch.Tensor,
    train_obj: torch.Tensor,
    train_con: torch.Tensor | None,
    bounds: torch.Tensor,
    pending_x: torch.Tensor | None,
) -> torch.Tensor:
    train_x = normalize(train_x, bounds=bounds)
    model = SaasFullyBayesianSingleTaskGP(
        train_x, train_obj,
        outcome_transform=Standardize(m=train_obj.size(-1)),
    )
    fit_fully_bayesian_model_nuts(
        model,
        warmup_steps=128,
        num_samples=256,
        thinning=16,
    )
    acqf = LogExpectedImprovement(model=model, best_f=train_obj.max())

    standard_bounds = torch.zeros_like(bounds)
    standard_bounds[1] = 1
    candidates, _ = optimize_acqf(
        acq_function=acqf,
        bounds=standard_bounds,
        q=1,
        num_restarts=10,
        raw_samples=512,
        options={"batch_limit": 5, "maxiter": 200},
    )
    return candidates


def _make_objective(tunable_features: set[str]):
    """Factory: returns an objective that only tunes the given features."""

    def objective(trial: optuna.Trial) -> float:
        fold = trial.number % K_FOLDS
        _random.seed(fold)

        weights = {}
        for feat in _ALL_FEATURES:
            if feat in tunable_features:
                weights[feat] = trial.suggest_float(f"w_{feat}", 0.0, 5.0)
            else:
                weights[feat] = 0.0

        active_weights = {f: w for f, w in weights.items() if w >= WEIGHT_THRESHOLD}
        active_features = list(active_weights.keys())

        if not active_features:
            return 0.0

        params = {
            "comparison_method": "ged",
            **_BEST_GED_PARAMS,
            "features": active_features,
            "property_normalizers": {f: _ALL_FEATURES[f] for f in active_features},
            "feature_weights": active_weights,
        }

        _, y_true, y_pred, _ = test_mnist_all(
            classes=list(classes_to_subclasses.keys()),
            params=params,
            sample_fraction=SAMPLE_FRACTION,
            description=f"trial {trial.number} | {len(active_features)} features",
            quiet=True,
            tracing_enabled=False,
        )
        accuracy = accuracy_score(y_true, y_pred)

        trial.set_user_attr("active_features", active_features)
        trial.set_user_attr("n_active", len(active_features))

        return accuracy

    return objective


def _run_phase(study_name, sampler, n_trials, phase_label, objective_fn):
    _storage = optuna.storages.RDBStorage(
        f"sqlite:///experiments/{study_name}.db"
    )
    study = optuna.create_study(
        study_name=study_name,
        storage=_storage,
        load_if_exists=True,
        direction="maximize",
        sampler=sampler,
    )
    completed = len([t for t in study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = max(0, n_trials - completed)
    print(f"[{phase_label}] {completed} done, {remaining} remaining.")

    if remaining > 0:
        study.optimize(objective_fn, n_trials=remaining, show_progress_bar=True)

    print(f"[{phase_label}] Best: {study.best_value*100:.2f}%")
    return study


def _extract_best_params(study):
    bp = study.best_params
    active_weights = {}
    for feat in _ALL_FEATURES:
        w = bp.get(f"w_{feat}", 0.0)
        if w >= WEIGHT_THRESHOLD:
            active_weights[feat] = w
    return {
        "comparison_method": "ged",
        **_BEST_GED_PARAMS,
        "features": list(active_weights.keys()),
        "property_normalizers": {f: _ALL_FEATURES[f] for f in active_weights},
        "feature_weights": active_weights,
    }


# ── Main ───────────────────────────────────────────────────
_original_experiment = evaluation.MLFLOW_EXPERIMENT
evaluation.MLFLOW_EXPERIMENT = OPTUNA_MLFLOW_EXPERIMENT

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5050"))
mlflow.set_experiment(OPTUNA_MLFLOW_EXPERIMENT)

try:
    with mlflow.start_run(run_name=f"optuna_{STUDY_NAME}") as parent_run:
        mlflow.log_params({
            "phase1_trials": PHASE1_TRIALS,
            "phase2_trials": PHASE2_TRIALS,
            "sample_fraction": SAMPLE_FRACTION,
            "k_folds": K_FOLDS,
            "weight_threshold": WEIGHT_THRESHOLD,
            "n_features": len(_ALL_FEATURES),
            "features": str(list(_ALL_FEATURES.keys())),
            "fixed_ged_params": str(_BEST_GED_PARAMS),
            "tuning": "weights only, Phase 2 constrained to SAASBO-selected features",
            "phase1_sampler": "BoTorchSampler + SAASBO (SaasFullyBayesianSingleTaskGP)",
            "phase2_sampler": "CmaEsSampler (popsize=16, constrained)",
        })

        # ── Phase 1: SAASBO discovery ──────────────────────
        print("=" * 60)
        print("PHASE 1: SAASBO — feature discovery (200 trials)")
        print("=" * 60)

        sampler_p1 = BoTorchSampler(
            candidates_func=saasbo_candidates_func,
            n_startup_trials=40,
            seed=42,
        )
        objective_p1 = _make_objective(set(_ALL_FEATURES.keys()))
        study_p1 = _run_phase(
            f"{STUDY_NAME}-saasbo", sampler_p1, PHASE1_TRIALS, "SAASBO",
            objective_fn=objective_p1,
        )

        mlflow.log_metric("phase1_best_accuracy", study_p1.best_value)
        best_p1 = _extract_best_params(study_p1)
        mlflow.log_metric("phase1_n_active_features", len(best_p1["features"]))
        print(f"Active features ({len(best_p1['features'])}): {best_p1['features']}")

        # ── Phase 2: CMA-ES refinement (constrained) ──────
        print()
        print("=" * 60)
        print("PHASE 2: CMA-ES — refinement (3000 trials)")
        print("=" * 60)

        active_from_saasbo = set(best_p1["features"])
        print(f"Constrained to {len(active_from_saasbo)} SAASBO-selected features:")
        for f in sorted(active_from_saasbo):
            print(f"  {f}")

        x0_cma = {f"w_{f}": study_p1.best_params.get(f"w_{f}", 0.0)
                   for f in active_from_saasbo}

        sampler_p2 = optuna.samplers.CmaEsSampler(
            x0=x0_cma,
            n_startup_trials=0,
            popsize=16,
            seed=42,
            warn_independent_sampling=False,
        )
        objective_p2 = _make_objective(active_from_saasbo)
        study_p2 = _run_phase(
            f"{STUDY_NAME}-cmaes-constrained", sampler_p2, PHASE2_TRIALS, "CMA-ES",
            objective_fn=objective_p2,
        )

        mlflow.log_metric("phase2_best_accuracy", study_p2.best_value)
        mlflow.log_metric("phase2_n_features", len(active_from_saasbo))

        # ── Pick overall best ──────────────────────────────
        if study_p2.best_value >= study_p1.best_value:
            best_study = study_p2
            best_phase = "CMA-ES"
        else:
            best_study = study_p1
            best_phase = "SAASBO"

        best_params = _extract_best_params(best_study)
        mlflow.log_metric("best_accuracy", best_study.best_value)

        try:
            importances = optuna.importance.get_param_importances(best_study)
            for param, imp in importances.items():
                mlflow.log_metric(f"importance.{param}", imp)
        except Exception:
            importances = {}

    # ── Summary ────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"Best accuracy: {best_study.best_value*100:.2f}% (from {best_phase})")
    print(f"\nActive features ({len(best_params['features'])}):")
    for f in best_params["features"]:
        w = best_params["feature_weights"][f]
        print(f"  {f:35s} w={w:.3f}")
    print(f"{'=' * 60}")

    if importances:
        print("\nParameter importance:")
        for param, imp in sorted(importances.items(), key=lambda x: -x[1]):
            bar = "█" * int(imp * 30)
            print(f"  {param:35s} {imp:.3f} {bar}")

    # ── Full-set validation ────────────────────────────────
    print(f"\n{'=' * 60}")
    print("Running full-set validation with best params...")
    print(json.dumps(best_params, indent=2))

    results, y_true, y_pred, run_dir = test_mnist_all(
        classes=list(classes_to_subclasses.keys()),
        params=best_params,
        sample_fraction=1.0,
        description=f"Full validation ({best_phase}, proxy: {best_study.best_value*100:.2f}%)",
        tracing_enabled=False,
    )

    full_accuracy = accuracy_score(y_true, y_pred)
    print(f"\nFull-dataset accuracy: {full_accuracy*100:.2f}%")
    print(f"Proxy accuracy (50%):  {best_study.best_value*100:.2f}%")
    print(f"Delta:                 {(full_accuracy - best_study.best_value)*100:+.2f}pp")

finally:
    evaluation.MLFLOW_EXPERIMENT = _original_experiment

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

delete_test_neo4j_nodes()

class_number = 1
img_num = 448
image_id = f"mnist_{class_number}_{img_num:05d}"
local_path = f"../../tests/generated_samples/mnist_{class_number}/test"
nuclio_path = f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"

img_file = f"{image_id}.png"
img_local = os.path.join(local_path, img_file)
if os.path.exists(img_local):
    img = mpimg.imread(img_local)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"MNIST Class {class_number}, Image #{img_num}")
    plt.axis("off")
    plt.show()

image_id_for_test = str(uuid.uuid4())
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "image_id": image_id_for_test,
    "session_id": "test",
    "delete_image_nodes": False,
    # "simplification_epsilon": 5,
}
result = classify_image(os.path.join(nuclio_path, img_file), params=params, timeout=60)
print(json.dumps(result, indent=2))

In [ ]:
# remove_concept("3_1")
retrain_concept(number=2, subclass=2, with_concept_creation=True)